# MediaPipe Gesture Recognizer ファインチューニング

このノートブックはゲスチャータイピングシステム用に、トレーニングデータを使用してMediaPipeモデルをファインチューニングします。

## ステップ1: 環境セットアップ

In [ ]:
!pip install mediapipe-model-maker numpy matplotlib pillow

## ステップ2: トレーニングデータの準備

GitHubからプロジェクトのトレーニングデータを取得するか、ローカルからアップロード

In [ ]:
import os
import shutil
from pathlib import Path
import random

# トレーニングデータのベースパス
# Colab環境の場合、Googleドライブをマウントしてアクセス
from google.colab import drive
drive.mount('/content/drive')

# ソースディレクトリ（ユーザーがアップロードしたデータ）
# または、GitHubから直接クローンすることも可能
source_base = '/content/drive/MyDrive/gesture_samples'  # 調整してください

## ステップ3: トレーニング/バリデーションデータの分割

80% トレーニング / 20% バリデーション

In [ ]:
# ディレクトリ構成の作成
train_dir = '/content/gesture_dataset/training'
val_dir = '/content/gesture_dataset/validation'

# ジェスチャーのクラス名
gestures = ['Zero', 'One', 'Two', 'Three', 'Four', 'Five', 'Thumb', 'None']

# ディレクトリの作成
for gesture in gestures:
    os.makedirs(f'{train_dir}/{gesture}', exist_ok=True)
    os.makedirs(f'{val_dir}/{gesture}', exist_ok=True)

print("ディレクトリ構成を作成しました")

## ステップ4: 画像データをトレーニング/バリデーションに分割

In [ ]:
# ソースからトレーニング/バリデーションディレクトリにコピー
# ファイル名の対応を調整 (例: zero-samples -> Zero)
gesture_mapping = {
    'zero-samples': 'Zero',
    'one-samples': 'One',
    'two-samples': 'Two',
    'three-samples': 'Three',
    'four-samples': 'Four',
    'five-samples': 'Five',
    'thumb-samples': 'Thumb',
    'null-samples': 'None'
}

# ハイパーパラメータ
TRAIN_RATIO = 0.8  # 80%をトレーニングに

for source_dir, gesture_class in gesture_mapping.items():
    source_path = os.path.join(source_base, source_dir)
    
    if not os.path.exists(source_path):
        print(f"警告: {source_path} が見つかりません")
        continue
    
    # ディレクトリ内の全ファイルを取得
    files = [f for f in os.listdir(source_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    random.shuffle(files)
    
    # トレーニング/バリデーション分割
    split_idx = int(len(files) * TRAIN_RATIO)
    train_files = files[:split_idx]
    val_files = files[split_idx:]
    
    # トレーニングデータをコピー
    for file in train_files:
        src = os.path.join(source_path, file)
        dst = os.path.join(train_dir, gesture_class, file)
        shutil.copy(src, dst)
    
    # バリデーションデータをコピー
    for file in val_files:
        src = os.path.join(source_path, file)
        dst = os.path.join(val_dir, gesture_class, file)
        shutil.copy(src, dst)
    
    print(f"{gesture_class}: {len(train_files)} train, {len(val_files)} validation")

print("\nデータセットの準備完了")

## ステップ5: MediaPipe Model Makerでモデルをトレーニング

In [ ]:
from mediapipe_model_maker import gesture_recognizer

# トレーニングデータの読み込み
train_data = gesture_recognizer.Dataset.from_folder(
    dirname=train_dir
)

validation_data = gesture_recognizer.Dataset.from_folder(
    dirname=val_dir
)

print("トレーニングデータを読み込みました")

## ステップ6: モデルのトレーニング実行

In [ ]:
# オプション設定
options = gesture_recognizer.GestureRecognizerOptions(
    base_model=gesture_recognizer.SupportedModels.MOBILENET_V2,
    num_epochs=100,
    batch_size=32,
    learning_rate=0.001
)

# モデルのトレーニング
model = gesture_recognizer.GestureRecognizer.create(
    training_data=train_data,
    validation_data=validation_data,
    options=options
)

print("モデルのトレーニング完了")

## ステップ7: モデルをエクスポート

In [ ]:
import os

# 出力ディレクトリの作成
output_dir = '/content/drive/MyDrive/gesture_model_output'
os.makedirs(output_dir, exist_ok=True)

# モデルをエクスポート
model.export(model_name='gesture_recognizer', model_dir=output_dir)

print(f"モデルを {output_dir} に保存しました")
print("\n出力ファイル:")
for file in os.listdir(output_dir):
    print(f"  - {file}")

## ステップ8: モデルをプロジェクトにコピー

トレーニング済みモデル（`gesture_recognizer.task`）を元のプロジェクトの `models/` ディレクトリにコピーしてください。